**Cell 1**

# NB17 — Metrics promised in main.tex, computed from the stored reward caches

**Purpose.** `main.tex` defines and promises several quantities that no run has reported
(review v3, section E). All of them follow from data already in the repository: every reward
cache stores each evaluated merge point `λ` with the per-prompt scores of every head. NB17
computes them for the three runs with reward data:

| run | regime | reward models |
|---|---|---|
| `nb10_rs_ppo` | RS-PPO, TinyLlama (ArmoRM was the training reward: circular) | ArmoRM |
| `nb13_dpo_seed137` | DPO, TinyLlama, seed 137 | ArmoRM |
| `nb13_1_mean_dpo` | mean-DPO over 5 seeds, TinyLlama | ArmoRM, URM |

| item | quantity | text reference |
|---|---|---|
| E1 | `R²_vert` (vertex-anchored, out of sample) and `R²_OLS` (best affine fit), per head, with CI; `√R²_OLS` = cap on any affine score | Def. `linearity-indicator`, Prop. `l2-cap` |
| E2 | Δm% against the specialists, uniform and p-weighted (ArmoRM only; URM has negative values) | Def. `delta-m` |
| E3 | win rate against `λ = p` per method configuration (CI over preferences), `ΔU_p / SE` | §3.7.4 |
| E4 | normalized weighted Tchebycheff shortfall `T_p^norm`, method vs. baseline | §3.7.3 |
| E5 | regret ratio `reg_p(λ*) / reg_p^random` | Def. `regret` |
| E6 | H-floor on the vertex matrix `M_ij = r_i(θ_j)`, global and p-aware (75 preferences), with bootstrap; R-floor for comparison; empirical check whether any evaluated point beats `λ = p` on every head | Rem. `h-limits`, Prop. `h-floor-collapse` |

**Uncertainty.** Paired bootstrap over the 80 prompts (the same prompt draw for every merge
point), 2000 draws (1000 for the method aggregates, 500 for E6), percentile 95 % intervals,
seed 1717.

**Scope notes for the thesis.**
* The evaluated set `B` is each run's Phase-B set (baselines of the 11 named preferences plus
  every moved method output, 137 to 219 points), not the 77-point search set of §6.4. The points
  cluster around the 11 preferences; this matters for E1 and E5.
* Cert is not in E2 to E5: on the collapsed floor it returns `λ = p` and does not move.
* No GPU, no model download, no Hugging Face token. About 5 minutes on any runtime.

**Script:** `scripts/promised_metrics.py` (new). **Output:** `results/nb17_promised_metrics/`.


**Cell 2**

Clone or update the repository. No Drive needed: all inputs are in the repository.


In [ ]:
# Cell 3
%cd /content
import os, shutil, sys
repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis
assert os.path.isfile("scripts/promised_metrics.py"), "Push scripts/promised_metrics.py first."


**Cell 4**

Settings and the computation (about 5 minutes on CPU).


In [ ]:
# Cell 5
from pathlib import Path
import importlib.util, json
import numpy as np, pandas as pd

PROJECT_ROOT = Path("/content/master-thesis").resolve()
OUT_DIR = PROJECT_ROOT / "results/nb17_promised_metrics"
N_BOOT = 2000
SEED = 1717

sys.path.insert(0, str(PROJECT_ROOT))
spec = importlib.util.spec_from_file_location("promised_metrics", PROJECT_ROOT / "scripts/promised_metrics.py")
pm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pm)

OUT = pm.run_all(PROJECT_ROOT, OUT_DIR, n_boot=N_BOOT, seed=SEED)
RESULTS = OUT["results"]
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
display(OUT["overview"].round(4))


**Cell 6**

**E1, linearity per head.** `R²_OLS` is the share of variance any affine rule in `λ` can
explain; `√R²_OLS` caps the Pearson correlation of every endpoint-linear score with that head
(Prop. `l2-cap`). `R²_vert` is what the reward-free vertex extrapolation achieves on the
non-vertex points; it can be negative.


In [ ]:
# Cell 7
for tag, res in RESULTS.items():
    print(f"\n=== {tag} ===")
    display(res["e1"].round(3))


**Cell 8**

**E2 to E5 by method family.** `win_rate`: share of (preference, configuration) rows with
`ΔU_p > 0`. `median_z`, `max_abs_z`: `ΔU_p / SE` over prompts. `mean_regret_ratio_method`:
regret relative to random selection on `B` (below 1 is better than random, compare with
`median_regret_ratio_baseline` for `λ = p`). `mean_t_norm_change > 0` and `share_t_norm_worse`:
worst-case shortfall worse than at `λ = p`. `mean_delta_m_change`: Δm% of the method minus Δm%
of `λ = p`, in percentage points. Intervals: paired bootstrap over prompts.


In [ ]:
# Cell 9
for tag, res in RESULTS.items():
    print(f"\n=== {tag} ===")
    display(res["summary"].round(4))


**Cell 10**

**E3, win rate per method configuration** over the evaluated preferences, with a bootstrap interval over preferences.


In [ ]:
# Cell 11
for tag, res in RESULTS.items():
    print(f"\n=== {tag} ===")
    display(res["win_rates"].round(3))


**Cell 12**

**E6, H-floor against R-floor.** The H-floor uses the measured vertex matrix
`M_ij = r_i(θ_j)` instead of `R`. It is open when some move along the simplex raises every head
in the linear vertex model. `bootstrap_share_global_open`: share of prompt resamples in which it
stays open. The empirical columns ask the same question of the actually evaluated merge points:
for how many preferences does some point beat `λ = p` on every head, as a point estimate and in
at least 95 % of the resamples.


In [ ]:
# Cell 13
rows = []
for tag, res in RESULTS.items():
    h = res["h_floor"]
    rows.append({"run": tag, "H global open": not h["global_collapsed"], "H global LP value": h["global_t"],
                 "H open preferences": f'{h["n_open"]}/{h["n_preferences"]}',
                 "H open in bootstrap": h["bootstrap_share_global_open"],
                 "R_cos open preferences": h.get("R_cos_n_open"),
                 "empirical: some point beats p on all heads": f'{h["empirical_prefs_with_dominating_point"]}/{h["empirical_n_prefs"]}',
                 "... in >= 95 % of resamples": f'{h["empirical_prefs_with_dominating_point_95"]}/{h["empirical_n_prefs"]}'})
display(pd.DataFrame(rows).set_index("run"))
names = list(pm.ATTRIBUTES)
for tag, res in RESULTS.items():
    print(f"\n{tag}: vertex matrix M (rows: reward head, columns: specialist)")
    display(pd.DataFrame(res["h_floor"]["vertex_matrix"], index=[f"r_{n}" for n in names],
                         columns=[f"e_{n}" for n in names]).round(4))


**Cell 14**

Export: the results folder as ZIP with repository paths.


In [ ]:
# Cell 15
import zipfile
zip_path = Path("/content/nb17_promised_metrics.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(PROJECT_ROOT).as_posix())
print(zip_path, f"{zip_path.stat().st_size / 1e3:.0f} kB")
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass
